<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🎯 Chapter 8 — Supervised Learning: Classification</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 75 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Understand the difference between binary and multi-class classification
- Train Logistic Regression, KNN, Decision Tree, and Naive Bayes classifiers
- Interpret classification metrics: accuracy, precision, recall, F1, AUC-ROC
- Read a confusion matrix and understand false positives vs false negatives
- Choose the right classifier for different business problems

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.naive_bayes     import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report
)

%matplotlib inline
np.random.seed(42)

# ── Dataset: Loan approval (binary classification) ────────────
n = 600
df = pd.DataFrame({
    'income':        np.random.randint(200000, 2000000, n),
    'loan_amount':   np.random.randint(100000, 5000000, n),
    'credit_score':  np.random.randint(300, 850, n),
    'employment':    np.random.choice([0, 1], n),  # 0=unemployed, 1=employed
    'dependents':    np.random.randint(0, 5, n),
})
# Create approval label with realistic signal
score = (
    0.4 * ((df['credit_score'] - 300) / 550)
    + 0.3 * (df['income'] / 2000000)
    + 0.2 * df['employment']
    - 0.1 * (df['loan_amount'] / 5000000)
    + np.random.normal(0, 0.15, n)
)
df['approved'] = (score > 0.35).astype(int)

print('Dataset shape:', df.shape)
print('Approval rate:', df['approved'].mean().round(2))
df.head()

## 📖 Section A — Classification Metrics

| Metric | Formula | When to Use |
|--------|---------|-------------|
| Accuracy | (TP+TN)/Total | Balanced classes |
| Precision | TP/(TP+FP) | Minimise false alarms (spam filter) |
| Recall | TP/(TP+FN) | Minimise missed cases (disease detection) |
| F1 | 2×P×R/(P+R) | Imbalanced classes, both FP and FN matter |
| AUC-ROC | Area under ROC curve | Overall discriminative ability |

> 💡 **Key Rule:** Accuracy is misleading on imbalanced datasets. If 95% of loans are approved, a model that always predicts 'approved' gets 95% accuracy — but it's useless. Use F1 or AUC-ROC instead.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Logistic Regression Baseline
# ─────────────────────────────────────────────────────────────
feature_cols = [c for c in df.columns if c != 'approved']
X = df[feature_cols].values
y = df['approved'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)
y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Rejected', 'Approved'], yticklabels=['Rejected', 'Approved'])
plt.title('Confusion Matrix — Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

---
## 🟢 Exercise 8.1 — Train and Compare Four Classifiers *(Level 1 · Est. 12 min)*

> Train LogisticRegression, KNN(k=5), DecisionTree, and GaussianNB. Print a comparison table of accuracy, F1, and AUC.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 8.1: Multi-Classifier Comparison
# ─────────────────────────────────────────────────────────────

classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Naive Bayes':         GaussianNB(),
}

# YOUR CODE HERE — train each, compute accuracy, F1, AUC, print comparison

results = {}
# ...

print(f'\n{"Model":22s}  {"Accuracy":>10} {"F1":>10} {"AUC":>10}')
print('-' * 58)
for name, m in results.items():
    print(f'{name:22s}  {m["accuracy"]:10.4f} {m["f1"]:10.4f} {m["auc"]:10.4f}')

In [ ]:
# @title ✅ Solution — Exercise 8.1 (click to expand)

classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Naive Bayes':         GaussianNB(),
}

results = {}
for name, clf in classifiers.items():
    clf.fit(X_train_s, y_train)
    yp   = clf.predict(X_test_s)
    prob = clf.predict_proba(X_test_s)[:, 1]
    results[name] = {
        'accuracy': accuracy_score(y_test, yp),
        'f1':       f1_score(y_test, yp),
        'auc':      roc_auc_score(y_test, prob),
    }

print(f'{"Model":22s}  {"Accuracy":>10} {"F1":>10} {"AUC":>10}')
print('-' * 58)
for name, m in results.items():
    print(f'{name:22s}  {m["accuracy"]:10.4f} {m["f1"]:10.4f} {m["auc"]:10.4f}')
print('\n✅ Solution verified! Decision Tree may overfit — its test score often drops vs train.')

---
## 🟡 Exercise 8.2 — ROC Curve and Threshold Tuning *(Level 2 · Est. 15 min)*

> Plot ROC curves for all four classifiers on the same figure. Then, for Logistic Regression, find the threshold that maximises F1 score.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 8.2: ROC Curves + Threshold Tuning
# ─────────────────────────────────────────────────────────────

# Plot all ROC curves on one figure
plt.figure(figsize=(8, 6))
# YOUR CODE HERE

# Find best threshold for Logistic Regression (maximise F1)
thresholds  = np.linspace(0.01, 0.99, 100)
lr_probs    = classifiers['Logistic Regression'].predict_proba(X_test_s)[:, 1]
best_thresh = # YOUR CODE HERE
best_f1     = # YOUR CODE HERE
print(f'Best threshold: {best_thresh:.2f}  Best F1: {best_f1:.4f}')

In [ ]:
# @title ✅ Solution — Exercise 8.2 (click to expand)

plt.figure(figsize=(8, 6))
for name, clf in classifiers.items():
    prob = clf.predict_proba(X_test_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves — Loan Approval Classifiers')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Threshold tuning
thresholds  = np.linspace(0.01, 0.99, 100)
lr_probs    = classifiers['Logistic Regression'].predict_proba(X_test_s)[:, 1]
f1_scores   = [f1_score(y_test, (lr_probs >= t).astype(int)) for t in thresholds]
best_thresh = thresholds[np.argmax(f1_scores)]
best_f1     = max(f1_scores)

print(f'Best threshold: {best_thresh:.2f}  Best F1: {best_f1:.4f}')
print('\n✅ The default threshold is 0.5 — but it is not always optimal!')
print('For fraud detection you may lower it (catch more fraud, accept more false alarms).')

---
## 🔴 Exercise 8.3 — Logistic Regression from Scratch *(Level 3 · Est. 25 min)*

> Implement `LogisticRegressionScratch` with `fit()` and `predict_proba()`. Use sigmoid activation and gradient descent. Verify accuracy matches sklearn's implementation.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 8.3: Logistic Regression from Scratch
# ─────────────────────────────────────────────────────────────

class LogisticRegressionScratch:
    def __init__(self, lr=0.1, epochs=500):
        self.lr = lr
        self.epochs = epochs

    def sigmoid(self, z):
        # YOUR CODE HERE — return 1 / (1 + exp(-z)), clip z to avoid overflow
        pass

    def fit(self, X, y):
        # YOUR CODE HERE — gradient descent on log loss
        pass

    def predict_proba(self, X):
        # YOUR CODE HERE
        pass

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

model_scratch = LogisticRegressionScratch(lr=0.1, epochs=500)
model_scratch.fit(X_train_s, y_train)
y_pred_scratch = model_scratch.predict(X_test_s)

acc_scratch  = accuracy_score(y_test, y_pred_scratch)
acc_sklearn  = accuracy_score(y_test, classifiers['Logistic Regression'].predict(X_test_s))
print(f'Scratch accuracy: {acc_scratch:.4f}')
print(f'Sklearn accuracy: {acc_sklearn:.4f}')

assert acc_scratch > 0.70, f'Expected accuracy > 0.70, got {acc_scratch:.4f}'
print('\n✅ Logistic regression from scratch verified!')

In [ ]:
# @title ✅ Solution — Exercise 8.3 (click to expand)

class LogisticRegressionScratch:
    def __init__(self, lr=0.1, epochs=500):
        self.lr, self.epochs = lr, epochs
        self.W, self.b = None, None

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)   # prevent overflow in exp
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n, p = X.shape
        self.W, self.b = np.zeros(p), 0.0
        for _ in range(self.epochs):
            y_hat = self.sigmoid(X @ self.W + self.b)
            err   = y_hat - y
            self.W -= self.lr * (1/n) * X.T @ err
            self.b -= self.lr * (1/n) * np.sum(err)

    def predict_proba(self, X):
        return self.sigmoid(X @ self.W + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

model_scratch = LogisticRegressionScratch(lr=0.1, epochs=500)
model_scratch.fit(X_train_s, y_train)
y_pred_scratch = model_scratch.predict(X_test_s)

acc_scratch = accuracy_score(y_test, y_pred_scratch)
acc_sklearn = accuracy_score(y_test, classifiers['Logistic Regression'].predict(X_test_s))
print(f'Scratch accuracy: {acc_scratch:.4f}')
print(f'Sklearn accuracy: {acc_sklearn:.4f}')

assert acc_scratch > 0.70
print('\n✅ The scratch implementation uses the same gradient update as sklearn — slight difference due to sklearn\'s L2 regularisation by default.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What is precision vs recall? When to prioritise each?</strong></summary>

**Answer:** Precision = of all positive predictions, how many were actually positive? (TP/(TP+FP)). Recall = of all actual positives, how many did we catch? (TP/(TP+FN)). High precision → fewer false alarms. High recall → fewer missed cases. Prioritise precision for spam filters (don't misclassify good emails). Prioritise recall for disease detection or fraud detection (never miss a real case, even if it means more false alarms). F1 is the harmonic mean — use it when both matter.
</details>

<details>
<summary><strong>Q2: What is AUC-ROC and what does a value of 0.85 mean?</strong></summary>

**Answer:** AUC (Area Under the ROC Curve) measures a classifier's ability to distinguish between classes across ALL possible thresholds. AUC = 0.5 means random guessing. AUC = 1.0 means perfect. AUC = 0.85 means: if you pick one positive and one negative example at random, the model ranks the positive higher with 85% probability. AUC is threshold-independent and class-imbalance robust — a better metric than accuracy for real-world binary classification.
</details>

<details>
<summary><strong>Q3: How does Logistic Regression output a class?</strong></summary>

**Answer:** Logistic Regression first computes a linear combination z = Wx + b, then squashes it through the sigmoid σ(z) = 1/(1+e^−z), producing a probability in (0, 1). The default decision threshold is 0.5: predict class 1 if σ(z) ≥ 0.5. Despite 'regression' in its name, it's a classification algorithm — the 'regression' refers to modelling the log-odds as a linear function.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 8 Complete!</h3>
<ul>
<li>Binary classification with 4 algorithms</li>
<li>Confusion matrix, precision, recall, F1, AUC-ROC</li>
<li>ROC curve plotting and threshold tuning</li>
<li>Logistic Regression from scratch via gradient descent</li>
</ul>
<p><strong>Next:</strong> Chapter 9 — Model Evaluation and Selection</p>
</div>